In [ ]:
import example_project as ep

ep.__version__

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
from jax.random import PRNGKey
from jax import numpy as jnp

pd.options.plotting.backend = "plotly"

from summer3.graph import *
from summer3.epi import Stratification, CompartmentMap, \
    CategoryGroup, StratSpec, mixing_matrix, ManagedArray, \
    CategoryData, TransitionFlow, CompartmentalEpiModel, \
    CompartmentalModelODE, strat_data_from_pandas, \
    build_istate, dti_to_epoch

In [ ]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
humans.compartments

In [ ]:
def new_iprocess(compartment_values: ManagedArray, contact_rate: float, infector_cats, infectious_compartments):
    ipops = compartment_values.sumcats(infector_cats.product(infectious_compartments))
    return contact_rate * ipops.sum() / compartment_values.sum()

In [ ]:
age_strat = humans.stratify(Stratification("age", ["child", "adult"]))

In [ ]:
age_cats = age_strat.categories()
infectees = age_cats
infectors = age_cats

In [ ]:
# defer is equivalent to using Function in the old summer2 style, eg the below code is equivalent to
# foi = Function(InfectionProcess.process, [iprocess, CompartmentValues, Parameter("contact_rate", 0.2)])

# Note also that Parameters now take default values

foi = defer(new_iprocess)(
    CompartmentValues, Parameter("contact_rate", 0.2), infectors, disease_state["I"],
)

In [ ]:
infection = TransitionFlow("infection", disease_state["S"], disease_state["I"], foi)
recovery = TransitionFlow(
    "recovery",
    disease_state["I"],
    disease_state["R"],
    1.0 / Parameter("recovery_time", 10.0),
)

In [ ]:
times = pd.date_range("7 jun 1980", "7 december 1980")
epi_model = CompartmentalEpiModel(humans, times)

epi_model.add_flow(infection)
epi_model.add_flow(recovery)

In [ ]:
pop_data = pd.Series(index=["child", "adult"], data=np.array([1000.0, 1500.0]))
base_pops = strat_data_from_pandas(pop_data, age_strat)
pop_splits = [CategoryData(disease_state.categories(), jnp.array(([0.9, 0.1, 0.0])))]

epi_model.set_initial_population(base_pops, pop_splits)

In [ ]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [ ]:
params = {"contact_rate": 0.2, "recovery_time": 20.0}
runner, istate = get_runner(epi_model)
results = epi_model.run(params)

In [ ]:
results["compartments"].to_pandas_df()

In [ ]:
inf_target = (
    results["flows"]["infection"]
    .sum(to_dims="time")
    .to_pandas_df()
    .rolling(7)
    .sum()[7:60:7]
)["data"]
inf_target_fuzzy = inf_target * np.exp(
    np.random.normal(scale=0.01, size=len(inf_target))
)
inf_target_fuzzy.plot()

In [ ]:
def get_derived_results(params):
    results = epi_model.run(params)  # runner.run(istate.data, params)
    inf_flow = results["flows"]["infection"]
    weekly_target = (
        inf_flow.sum(to_dims="time").rolling(7, jnp.sum).query(time=inf_target.index)
    )
    return weekly_target